# ⚙️ 02 - Data Preprocessing & Feature Engineering

Thực hiện tiền xử lý dữ liệu và trích xuất đặc trưng số học theo yêu cầu đề bài:
- **1. Làm sạch văn bản**: Loại bỏ stop words, punctuation, HTML tags/URLs, chuẩn hóa Unicode
- **2. Phân chia tập dữ liệu độc lập** (Early Stratified Split 80/10/10) chống rò rỉ dữ liệu
- **3. Vector hóa văn bản bằng TF-IDF** (TfidfVectorizerScratch - Unigram & Bigram)
- **4. Trích xuất đặc trưng hình thức** (Tần suất ký tự '!', '$', chữ in hoa, độ dài)
- **5. Ghép nối không gian ma trận lai** (Hybrid Feature Matrix: Word + Char + Keywords + Numerics)
- **6. Thử nghiệm các tổ hợp đặc trưng** (Feature Combinations Experiment)


Notebook này **không gọi trực tiếp các model có sẵn** như `sklearn.linear_model`,
`sklearn.svm`, `sklearn.naive_bayes`, `sklearn.ensemble`, `sklearn.model_selection`
hoặc `sklearn.feature_extraction`.

Các thuật toán được gọi từ package tự xây dựng của project:

```text
src/
├── data/
├── features/
├── models/
├── evaluation/
└── deployment/
```

Các thư viện tính toán nền tảng (NumPy/Pandas/Scipy/Matplotlib...) nếu có được sử dụng
thì được **ẩn bên trong `src/`**, còn notebook chỉ làm việc với API của project.

In [ ]:
# LOGIC IMPORT THƯ VIỆN & CÁC MODULE TIỀN XỬ LÝ:
# ---------------------------------------------
# 1. Thêm project root vào sys.path.
# 2. Import numpy, pandas, scipy.sparse.
# 3. Import src.data.preprocessing (stratified_split_dataframe, clean_email_split, SpamTextProcessor).
# 4. Import src.features.feature_engineering (TfidfVectorizerScratch, MaxAbsScalerScratch, HybridFeatureBuilderScratch).
pass


In [2]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\hop98\Documents\Huyền\SpamClassifier_My_Task1\spam_classifier-main"
)

print("Project tồn tại:", PROJECT_ROOT.exists())
print("Thư mục src tồn tại:", (PROJECT_ROOT / "src").is_dir())

if PROJECT_ROOT.exists():
    print("Các mục trong project:", os.listdir(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print("Thư mục làm việc:", os.getcwd())

Project tồn tại: True
Thư mục src tồn tại: True
Các mục trong project: ['.agents', '.gitignore', 'CACH_CHAY_TASK1.txt', 'data', 'GIAI_THICH_TASK1.md', 'KET_QUA_KIEM_TRA.json', 'kiem_tra_task1.py', 'main.py', 'notebooks', 'README.md', 'requirements.txt', 'results', 'src', 'tests']
Thư mục làm việc: C:\Users\hop98\Documents\Huyền\SpamClassifier_My_Task1\spam_classifier-main


In [5]:
from src.data.loader import load_raw_data
from src.data.preprocessing import (
    stratified_split_dataframe,
    clean_email_split,
    SpamTextProcessor
)
from src.features.feature_engineering import (
    TfidfVectorizerScratch,
    HybridFeatureBuilderScratch
)

print("Import thành công!")

Import thành công!


In [7]:
import pandas as pd

df_raw = load_raw_data("data/raw/spam.csv")

# Chuyển list thành DataFrame nếu cần
if isinstance(df_raw, list):
    df_raw = pd.DataFrame(df_raw)

# Chuẩn hóa tên cột nếu dữ liệu dùng định dạng SMS Spam Collection
if "label" not in df_raw.columns and "v1" in df_raw.columns:
    df_raw = df_raw.rename(columns={"v1": "label", "v2": "text"})

print("Dataset:", df_raw.shape)
print("Các cột:", df_raw.columns.tolist())
display(df_raw.head())

Dataset: (43849, 4)
Các cột: ['source_id', 'label', 'text', 'raw_text']


,source_id,label,text,raw_text
0,1,ham,"Go until jurong point, crazy.. Available only ...","Go until jurong point, crazy.. Available only ..."
1,2,ham,Ok lar... Joking wif u oni...,Ok lar... Joking wif u oni...
2,3,spam,Free entry in 2 a wkly comp to win FA Cup fina...,Free entry in 2 a wkly comp to win FA Cup fina...
3,4,ham,U dun say so early hor... U c already then say...,U dun say so early hor... U c already then say...
4,5,ham,"Nah I don't think he goes to usf, he lives aro...","Nah I don't think he goes to usf, he lives aro..."


## 1. Phân chia tập dữ liệu độc lập (Early Stratified Split)


In [ ]:
# LOGIC PHÂN TẦNG DỮ LIỆU ĐỘC LẬP TRÁNH RÒ RỈ DỮ LIỆU:
# -----------------------------------------------------
# 1. Nạp df_raw từ data/raw/spam.csv.
# 2. Gọi stratified_split_dataframe() chia thành: train_df (80%), val_df (10%), test_df (10%).
# 3. Kiểm tra tỷ lệ Spam/Ham trên cả 3 tập để xác nhận tính đồng nhất.
pass


In [9]:
import inspect

print("Chữ ký hàm:")
print(inspect.signature(stratified_split_dataframe))

print("\nCode của hàm:")
print(inspect.getsource(stratified_split_dataframe))

Chữ ký hàm:
(df, label_col='label', train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, random_state=42)

Code của hàm:
def stratified_split_dataframe(df, label_col='label', train_ratio=0.80,
                               val_ratio=0.10, test_ratio=0.10, random_state=42):
    """Chia phân tầng tự viết bằng Random; trả train, val, test là list[dict].

    Làm tròn bằng phần dư lớn nhất, nên tổng số mẫu được bảo toàn. Với lớp
    quá nhỏ, không thể bảo đảm lớp xuất hiện ở mọi tập; báo lỗi rõ ràng.
    """
    ratios = (train_ratio, val_ratio, test_ratio)
    if any(not math.isfinite(r) or r < 0 for r in ratios) or not math.isclose(sum(ratios), 1.0, abs_tol=1e-9):
        raise ValueError('Tỷ lệ chia phải không âm, hữu hạn và có tổng bằng 1.')
    if not df:
        raise ValueError('Dữ liệu chia tập không được rỗng.')
    groups = {}
    content_keys = set()
    for row in df:
        label = _normalise_label(row.get(label_col))
        if label not in ('ham', 'spam'):
            raise Val

In [10]:

# 1. Nạp dữ liệu gốc
df_raw = load_raw_data("data/raw/spam.csv")

# Chuyển DataFrame thành list[dict] nếu cần
if isinstance(df_raw, pd.DataFrame):
    df_raw = df_raw.to_dict("records")

# Chuẩn hóa tên cột nếu dữ liệu dùng v1, v2
if df_raw and "v1" in df_raw[0] and "label" not in df_raw[0]:
    for row in df_raw:
        row["label"] = row.pop("v1")
        row["text"] = row.pop("v2")

# 2. Làm sạch dữ liệu trước khi chia tập
clean_df = clean_email_split(df_raw)

print("Số mẫu sau khi làm sạch:", len(clean_df))

# 3. Chia phân tầng: 80% train, 10% validation, 10% test
train_df, val_df, test_df = stratified_split_dataframe(
    clean_df,
    label_col="label",
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
    random_state=42
)

# 4. Kiểm tra số lượng và tỷ lệ nhãn
for name, data in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    labels = pd.Series([row["label"] for row in data])

    print(f"\n===== {name} =====")
    print("Số lượng mẫu:", len(data))
    print("Tỷ lệ nhãn (%):")
    print(
        (labels.value_counts(normalize=True) * 100)
        .round(2)
        .to_string()
    )

# 5. Kiểm tra tổng số mẫu
print("\n===== Kiểm tra tổng dữ liệu =====")
print("Dữ liệu sau làm sạch:", len(clean_df))
print("Tổng 3 tập:", len(train_df) + len(val_df) + len(test_df))

Số mẫu sau khi làm sạch: 29028

===== Train =====
Số lượng mẫu: 23223
Tỷ lệ nhãn (%):
ham     57.31
spam    42.69

===== Validation =====
Số lượng mẫu: 2903
Tỷ lệ nhãn (%):
ham     57.32
spam    42.68

===== Test =====
Số lượng mẫu: 2902
Tỷ lệ nhãn (%):
ham     57.31
spam    42.69

===== Kiểm tra tổng dữ liệu =====
Dữ liệu sau làm sạch: 29028
Tổng 3 tập: 29028


## 2. Làm sạch văn bản (Cleaning: Stop Words, Punctuation, HTML Tags)


In [ ]:
# LOGIC LÀM SẠCH VĂN BẢN VÀ LỌC MẪU BẤT THƯỜNG:
# -----------------------------------------------
# 1. Gọi clean_email_split() cho từng tập riêng biệt để lọc dòng rỗng, null, duplicate.
# 2. Khởi tạo SpamTextProcessor thực hiện chuẩn hóa văn bản:
#    - Chuyển lowercase, chuẩn hóa Unicode NFKD.
#    - Xóa HTML tags và chuyển URL thành <URL>.
#    - Chuyển số tiền thành <MONEY>, số điện thoại thành <NUMBER>.
#    - Loại bỏ dấu câu và stop words.
# 3. So sánh văn bản trước và sau làm sạch trên 5 email mẫu.
pass


In [11]:
import inspect

print(inspect.signature(SpamTextProcessor))
print([
    name for name in dir(SpamTextProcessor)
    if not name.startswith("_")
])

(config: Any | None = None)
['extract_char_ngrams', 'extract_structural_features', 'extract_word_ngrams', 'normalize_text', 'prepare_feature_frame', 'tokenize_words']


In [13]:
import inspect

print("Các phương thức của SpamTextProcessor:")
print([
    name for name in dir(processor)
    if not name.startswith("_") and callable(getattr(processor, name))
])

print("\nCode của lớp:")
print(inspect.getsource(SpamTextProcessor))

Các phương thức của SpamTextProcessor:
['extract_char_ngrams', 'extract_structural_features', 'extract_word_ngrams', 'normalize_text', 'prepare_feature_frame', 'tokenize_words']

Code của lớp:
class SpamTextProcessor:
    """Bộ xử lý không trạng thái học; ngưỡng cắt độ dài do builder fit trên Train."""
    def __init__(self, config: Optional[Any] = None):
        self.config = config

    def normalize_text(self, text: str) -> str:
        return clean_text_basic(text)

    def tokenize_words(self, text: str) -> list[str]:
        return text.split()

    def extract_word_ngrams(self, tokens: list[str], n: int = 2) -> list[str]:
        if not isinstance(n, int) or n < 1:
            raise ValueError('Bậc n-gram phải là số nguyên dương.')
        return [' '.join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

    def extract_char_ngrams(self, text: str, min_n: int = 3, max_n: int = 5) -> list[str]:
        if min_n < 1 or max_n < min_n:
            raise ValueError('Khoảng char

In [14]:

# 1. Khởi tạo bộ xử lý văn bản
processor = SpamTextProcessor()

# 2. Chọn 5 email mẫu từ tập train
sample_emails = train_df[:5]

# 3. So sánh văn bản trước và sau khi chuẩn hóa
for i, email in enumerate(sample_emails, start=1):
    original_text = str(email.get("raw_text", email.get("text")) or "")

    # Sử dụng đúng phương thức có trong SpamTextProcessor
    cleaned_text = processor.normalize_text(original_text)

    print(f"\n===== EMAIL {i} =====")
    print("Nhãn:", email.get("label", "Không xác định"))

    print("\nTrước khi làm sạch:")
    print(original_text)

    print("\nSau khi làm sạch:")
    print(cleaned_text)

    print("-" * 60)


===== EMAIL 1 =====
Nhãn: ham

Trước khi làm sạch:
Maybe you should find something else to do instead???

Sau khi làm sạch:
maybe find something else instead
------------------------------------------------------------

===== EMAIL 2 =====
Nhãn: ham

Trước khi làm sạch:
How are you enjoying this semester? Take care brother.

Sau khi làm sạch:
how enjoying semester take care brother
------------------------------------------------------------

===== EMAIL 3 =====
Nhãn: spam

Trước khi làm sạch:
 escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd cbs news sts escapenumber status repo

## 3. Vector hóa văn bản bằng TF-IDF (TfidfVectorizerScratch)


In [ ]:
# LOGIC BIẾN ĐỔI VĂN BẢN THÀNH MA TRẬN TF-IDF:
# --------------------------------------------
# 1. Khởi tạo TfidfVectorizerScratch(min_df=2, max_features=3000, sublinear_tf=True).
# 2. Fit trên train_df['model_text'] và transform trên val_df, test_df.
# 3. Kiểm tra kích thước ma trận thưa và danh sách các từ vựng có IDF cao nhất.
pass


In [18]:

# Khởi tạo bộ xử lý văn bản
processor = SpamTextProcessor()

# Tạo model_text cho từng tập
train_df = processor.prepare_feature_frame(train_df)
val_df = processor.prepare_feature_frame(val_df)
test_df = processor.prepare_feature_frame(test_df)

# Kiểm tra kết quả
print("Train:", train_df[0].keys())
print("Văn bản mẫu:", train_df[0]["model_text"][:200])

Train: dict_keys(['source_id', 'label', 'text', 'raw_text', 'has_url', 'has_number', 'digit_count', 'special_char_count', 'exclamation_count', 'dollar_count', 'question_count', 'uppercase_ratio', 'raw_length', 'raw_word_count', 'model_text', 'model_length', 'y'])
Văn bản mẫu: maybe find something else instead


In [20]:
X_train_text = [row["model_text"] for row in train_df]
X_val_text = [row["model_text"] for row in val_df]
X_test_text = [row["model_text"] for row in test_df]

In [22]:
import inspect

print("Các thuộc tính của TF-IDF:")
print([
    name for name in dir(tfidf)
    if not name.startswith("__")
])

print("\nCode của lớp TfidfVectorizerScratch:")
print(inspect.getsource(TfidfVectorizerScratch))

Các thuộc tính của TF-IDF:
['_tokens', 'fit', 'fit_transform', 'idf_diag_', 'max_features', 'min_df', 'norm', 'smooth_idf', 'sublinear_tf', 'transform', 'vocabulary_']

Code của lớp TfidfVectorizerScratch:
class TfidfVectorizerScratch:
    """DF đếm số tài liệu có token, không phải tổng số lần token xuất hiện.

    Ngoài chuỗi, nhận danh sách token để giữ nguyên bigram/char n-gram có
    khoảng trắng. Không nối n-gram rồi split, vì sẽ phá vỡ đơn vị đặc trưng.
    """
    def __init__(self, min_df=2, max_features=None, sublinear_tf=True,
                 smooth_idf=True, norm='l2'):
        if not isinstance(min_df, int) or min_df < 1:
            raise ValueError('min_df phải là số nguyên dương.')
        if max_features is not None and (not isinstance(max_features, int) or max_features < 1):
            raise ValueError('max_features phải là số nguyên dương hoặc None.')
        if norm not in ('l2', None):
            raise ValueError('norm chỉ hỗ trợ l2 hoặc None.')
        self.min_

In [23]:

import numpy as np

# 4. Lấy danh sách từ vựng và giá trị IDF
vocabulary = tfidf.vocabulary_
idf_values = np.asarray(tfidf.idf_diag_)

# Sắp xếp các từ theo IDF giảm dần
top_indices = np.argsort(idf_values)[::-1][:20]

print("===== 20 TỪ CÓ IDF CAO NHẤT =====")

if len(top_indices) == 0:
    print("Từ vựng rỗng, không có IDF để hiển thị.")
else:
    # Đổi chỉ số thành từ tương ứng
    index_to_word = {index: word for word, index in vocabulary.items()}

    for index in top_indices:
        word = index_to_word[index]
        print(f"{word}: {idf_values[index]:.4f}")

===== 20 TỪ CÓ IDF CAO NHẤT =====
intelligence: 6.3080
images: 6.3080
gray: 6.3080
gerald: 6.3080
execution: 6.3080
escapenumbert: 6.3080
escapenumberc: 6.3080
equivalent: 6.3080
ending: 6.3080
ended: 6.3080
contacting: 6.3080
closer: 6.3080
blog: 6.3080
agenda: 6.3080
affected: 6.3080
usual: 6.2994
shape: 6.2994
shaescapenumber: 6.2994
overview: 6.2994
medium: 6.2994


## 4. Xây dựng ma trận đặc trưng lai toàn diện & thử nghiệm tổ hợp đặc trưng


In [ ]:
# LOGIC GHÉP NỐI MA TRẬN ĐẶC TRƯNG LAI (HYBRID FEATURES):
# --------------------------------------------------------
# 1. Khởi tạo HybridFeatureBuilderScratch ghép [Word TF-IDF + Char N-grams + Keywords + Scaled Numerics].
# 2. Fit trên train_df và transform trên val_df, test_df thu được X_train_csr, X_val_csr, X_test_csr.
# 3. Chạy experiment_feature_combinations() thử nghiệm 4 tổ hợp đặc trưng khác nhau
#    để so sánh đóng góp của từng nhóm thuộc tính.
pass


In [24]:

import inspect

print("===== HybridFeatureBuilderScratch =====")
print("Constructor:", inspect.signature(HybridFeatureBuilderScratch))

print("\nCác phương thức:")
print([
    name for name in dir(HybridFeatureBuilderScratch)
    if not name.startswith("_")
])

print("\n===== experiment_feature_combinations =====")
print("Đã import hàm chưa:", "experiment_feature_combinations" in globals())

if "experiment_feature_combinations" in globals():
    print(inspect.signature(experiment_feature_combinations))
    print(inspect.getsource(experiment_feature_combinations))

===== HybridFeatureBuilderScratch =====
Constructor: (config: Any, text_processor: Any, signal_extractor: src.features.feature_engineering.SpamSignalFeatureExtractor)

Các phương thức:
['fit', 'fit_transform', 'transform']

===== experiment_feature_combinations =====
Đã import hàm chưa: False


In [25]:

print("===== Code HybridFeatureBuilderScratch =====")
print(inspect.getsource(HybridFeatureBuilderScratch))

===== Code HybridFeatureBuilderScratch =====
class HybridFeatureBuilderScratch:
    """Một fit duy nhất trên Train; mọi transform giữ nguyên thứ tự cột.

    feature_slices_ ghi (start, stop) cho từng khối để mô hình/phân tích lỗi
    xác định nguồn đặc trưng. Tất cả đặc trưng không âm, dùng được cho NB.
    """
    def __init__(self, config: Any, text_processor: Any, signal_extractor: SpamSignalFeatureExtractor):
        self.config, self.text_processor, self.signal_extractor = config, text_processor, signal_extractor
        self.word_vectorizer = self.bigram_vectorizer = self.char_vectorizer = None
        self.numeric_scaler = None
        self.feature_names_, self.feature_slices_ = [], {}
        self.cap_word_limit_ = None
        self.params_ = None
        self.fitted_config_ = None
        self.fitted_signal_extractor_ = None
        self.is_fitted_ = False

    def _prepare(self, records):
        return self.text_processor.prepare_feature_frame(records, self.cap_word_limit_,